# ARC-AGI-3 Live Visualization

Ejecución paso a paso con visualización en tiempo real.

In [ ]:
# Setup
import sys
sys.path.insert(0, '/root/arc-agi-3/kaggle-starter/agent')
sys.path.insert(0, '/root/arc-agi-3/kaggle-starter/vendor/ARC-AGI-3-Agents')

import json
import time
import random
import subprocess
import arc_agi
from arc_agi import OperationMode
from arcengine import GameAction, GameState, FrameData

from socratic_agent import SocraticAgent, ActionRecord
from arc3_memory import PatternMemory
from arc3_feedback import AlgedonicFeedback, SignalType
from arc3_canon import StateCanon

print('✓ Modules loaded')

In [ ]:
# Visualizer
from IPython.display import display, HTML, clear_output
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import ListedColormap
from collections import Counter

COLORS = ['#1a1a2e', '#e94560', '#0f3460', '#16213e', 
          '#533483', '#e94560', '#00b4d8', '#ffbe0b']
cmap = ListedColormap(COLORS)

class LiveVisualizer:
    def __init__(self):
        self.fig, self.axes = plt.subplots(2, 2, figsize=(14, 10))
        self.fig.suptitle('ARC-AGI-3 Socratic Agent — Live', fontsize=14)
        plt.ion()
        plt.show(block=False)
        self.fig.canvas.draw()
        self.fig.canvas.flush_events()
    
    def update(self, grid, action_history, current_action, feedback, socratic, step, state):
        # Clear all axes
        for ax in self.axes.flat:
            ax.clear()
        
        # 1. Grid
        ax = self.axes[0, 0]
        if grid and isinstance(grid, list):
            arr = np.array(grid)
            ax.imshow(arr, cmap=cmap, interpolation='nearest')
            for i in range(arr.shape[0]):
                for j in range(arr.shape[1]):
                    ax.text(j, i, str(arr[i,j]), ha='center', va='center', 
                            color='white', fontsize=8)
        ax.set_title(f'Step {step}: {current_action} → {state}')
        
        # 2. Actions
        ax = self.axes[0, 1]
        if action_history:
            counts = Counter(action_history)
            actions = list(counts.keys())
            values = list(counts.values())
            colors = ['#00b4d8' if a != current_action else '#e94560' for a in actions]
            ax.bar(actions, values, color=colors)
        ax.set_title(f'Actions ({len(action_history)} total)')
        ax.tick_params(axis='x', rotation=45)
        
        # 3. Feedback
        ax = self.axes[1, 0]
        signals = feedback.get_recent_signals(20)
        if signals:
            vals = []
            cols = []
            for s in signals:
                if s.signal_type == SignalType.PLEASURE:
                    vals.append(s.intensity)
                    cols.append('#00b4d8')
                elif s.signal_type == SignalType.PAIN:
                    vals.append(-s.intensity)
                    cols.append('#e94560')
                else:
                    vals.append(0)
                    cols.append('#533483')
            ax.bar(range(len(vals)), vals, color=cols)
        ax.axhline(y=0, color='gray', linestyle='--', linewidth=0.5)
        ax.set_title(f'Reward: {feedback.get_reward():.2f}')
        
        # 4. Memory
        ax = self.axes[1, 1]
        if socratic.action_scores:
            acts = list(socratic.action_scores.keys())
            scores = list(socratic.action_scores.values())
            cols = ['#00b4d8' if s > 0 else '#e94560' for s in scores]
            ax.barh(acts, scores, color=cols)
        ax.axvline(x=0, color='gray', linestyle='--', linewidth=0.5)
        ax.set_title('Socratic Scores')
        
        plt.tight_layout()
        self.fig.canvas.draw()
        self.fig.canvas.flush_events()
        plt.pause(0.05)

print('✓ Visualizer defined')

In [ ]:
# LLM helper
OLLAMA_URL = 'http://localhost:11434/api/chat'
MODEL = 'qwen3:1.7b'
ACTION_MAP = {
    'ACTION1': GameAction.ACTION1, 'ACTION2': GameAction.ACTION2,
    'ACTION3': GameAction.ACTION3, 'ACTION4': GameAction.ACTION4,
    'ACTION5': GameAction.ACTION5, 'ACTION6': GameAction.ACTION6,
    'ACTION7': GameAction.ACTION7,
}

def ask_llm(prompt):
    payload = json.dumps({
        'model': MODEL,
        'messages': [{'role': 'user', 'content': prompt}],
        'stream': False,
        'think': False,
        'options': {'num_predict': 200, 'num_ctx': 2048}
    })
    try:
        result = subprocess.run(
            ['curl', '-s', OLLAMA_URL, '-d', payload],
            capture_output=True, text=True, timeout=30
        )
        data = json.loads(result.stdout)
        return data.get('message', {}).get('content', '')
    except:
        return ''

def parse_action(response):
    response = response.strip().upper()
    for name, action in ACTION_MAP.items():
        if name in response:
            return action
    return None

print('✓ LLM helper defined')

In [ ]:
# Game runner with live visualization
def run_game_live(game_id='ls20', max_steps=20):
    # Initialize modules
    socratic = SocraticAgent(game_id)
    memory = PatternMemory(game_id)
    feedback = AlgedonicFeedback(game_id)
    canon = StateCanon(game_id)
    
    # Create game env directly
    arc = arc_agi.Arcade(operation_mode=OperationMode.NORMAL)
    env = arc.make(game_id)
    
    # Create visualizer
    viz = LiveVisualizer()
    
    action_history = []
    prev_frame = None
    
    print(f'=== Running {game_id} ({max_steps} steps) ===')
    print('Close the plot window to stop early.\n')
    
    # Reset game
    env.reset()
    frame = env.step(GameAction.RESET)
    
    try:
        for step in range(max_steps):
            state = frame.state
            grid = frame.frame
            levels = frame.levels_completed if hasattr(frame, 'levels_completed') else 0
            
            # If game over or not played, reset
            if state in (GameState.NOT_PLAYED, GameState.GAME_OVER):
                action = GameAction.RESET
                action.reasoning = 'reset'
                action_name = 'RESET'
            elif state == GameState.WIN:
                print(f'\n🎉 WIN at step {step}!')
                break
            else:
                # === PIPELINE ===
                grid_data = {
                    'state': str(state),
                    'grid': grid,
                    'levels_completed': levels,
                }
                
                # Canon
                if prev_frame is not None:
                    prev_data = {
                        'state': str(prev_frame.state),
                        'grid': prev_frame.frame,
                        'levels_completed': prev_frame.levels_completed if hasattr(prev_frame, 'levels_completed') else 0,
                    }
                    declared = canon.predict(prev_data)
                    observed = canon.observe(grid_data)
                    canon.compare(declared, observed)
                
                # Memory
                detected = memory.detect_patterns(grid_data)
                pattern_ctx = memory.to_prompt_fragment(detected)
                
                # Feedback
                feedback_ctx = feedback.to_prompt_fragment()
                
                # Canon context
                canon_ctx = canon.to_prompt_fragment()
                
                # Socratic
                candidate_actions = [a.name for a in GameAction 
                                   if a is not GameAction.RESET and not a.is_complex()]
                socratic_hint = socratic.suggest_action(str(state), candidate_actions)
                
                # LLM prompt
                tried = Counter(action_history[-20:]) if action_history else {}
                
                prompt = f'''Game "{game_id}" — Step {step+1}
Current: state={state}, levels={levels}
Available: {', '.join(candidate_actions)}
Recently tried: {dict(tried)}
TRY A DIFFERENT ACTION if possible.
Suggestion: {socratic_hint}
{pattern_ctx}
{feedback_ctx}
Choose ONE action. Reply ONLY with action name.'''
                
                response = ask_llm(prompt)
                action = parse_action(response)
                
                if action is None:
                    if socratic_hint and socratic_hint in ACTION_MAP:
                        action = ACTION_MAP[socratic_hint]
                    else:
                        action = random.choice([a for a in GameAction 
                                               if a is not GameAction.RESET and not a.is_complex()])
                
                if action.is_complex():
                    action.set_data({'x': 0, 'y': 0})
                
                action_name = action.name
                
                # Record for socratic
                record = ActionRecord(
                    action=action_name,
                    state_before=str(state),
                    state_after=str(state),
                    levels_before=levels,
                    levels_after=levels,
                    game_over=False,
                    won=False,
                    timestamp=time.time(),
                    llm_reasoning=response[:200] if response else '',
                    pattern_used=detected[0] if detected else '',
                )
                socratic.record_action(record)
                
                # Feedback
                if prev_frame is not None:
                    feedback.evaluate(
                        str(prev_frame.state), str(state),
                        prev_frame.levels_completed if hasattr(prev_frame, 'levels_completed') else 0,
                        levels,
                        action_name
                    )
                
                action_history.append(action_name)
            
            # Execute action
            frame = env.step(action)
            
            # Update visualization
            viz.update(
                grid=frame.frame if hasattr(frame, 'frame') else None,
                action_history=action_history,
                current_action=action_name,
                feedback=feedback,
                socratic=socratic,
                step=step+1,
                state=str(frame.state)
            )
            
            # Print step
            llm_info = ''
            if isinstance(action.reasoning, dict):
                llm_info = action.reasoning.get('llm_raw', '')[:50]
            print(f'Step {step+1:2d}: {action_name:8s} → {str(frame.state):15s} | LLM: {llm_info}')
            
            prev_frame = frame
            time.sleep(0.05)
    
    except KeyboardInterrupt:
        print('\n\nStopped by user.')
    
    plt.ioff()
    plt.show()
    
    # Summary
    print(f'\n=== Summary ===')
    print(f'Steps: {len(action_history)}')
    print(f'Unique actions: {len(set(action_history))}')
    print(f'Reward: {feedback.get_reward():.2f}')
    print(f'Canon accuracy: {canon.get_accuracy():.0%}')
    print(f'Action scores: {socratic.action_scores}')
    
    return socratic, memory, feedback, canon, action_history

print('✓ Game runner defined')

In [ ]:
# Run!
random.seed(42)
socratic, memory, feedback, canon, history = run_game_live('ls20', max_steps=15)

In [ ]:
# Summary plot after run
if 'history' in dir() and history:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    counts = Counter(history)
    axes[0].bar(counts.keys(), counts.values(), color='#00b4d8')
    axes[0].set_title('Action Distribution')
    axes[0].tick_params(axis='x', rotation=45)
    
    rewards = []
    for s in feedback.signals:
        if s.signal_type == SignalType.PLEASURE:
            rewards.append(rewards[-1] + s.intensity if rewards else s.intensity)
        elif s.signal_type == SignalType.PAIN:
            rewards.append(rewards[-1] - s.intensity if rewards else -s.intensity)
        else:
            rewards.append(rewards[-1] if rewards else 0)
    axes[1].plot(rewards, color='#00b4d8', linewidth=2)
    axes[1].axhline(y=0, color='gray', linestyle='--')
    axes[1].set_title('Cumulative Reward')
    
    if socratic.action_scores:
        acts = list(socratic.action_scores.keys())
        scores = list(socratic.action_scores.values())
        cols = ['#00b4d8' if s > 0 else '#e94560' for s in scores]
        axes[2].barh(acts, scores, color=cols)
        axes[2].axvline(x=0, color='gray', linestyle='--')
        axes[2].set_title('Socratic Scores')
    
    plt.tight_layout()
    plt.savefig('/root/arc-agi-3/kaggle-starter/results/evaluation.png', dpi=150)
    plt.show()
    print('✓ Saved to results/evaluation.png')